**Mini-Project - Streamlit App for EDA with Starbucks Data**


# PART 1: Data Cleaning

Before diving into the modeling process, it's crucial to ensure that your data is clean and ready for analysis.

- **Check null values:** Examine the dataset for any missing values. Addressing null values is essential to prevent issues during model training and evaluation.

- **Check dtypes:** Ensure that the data types of your features are appropriate. This step is important for avoiding potential discrepancies between the expected and actual data types.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay

In [2]:
stb = pd.read_csv('starbucks.csv')

In [3]:
stb.head()

,Beverage_category,Beverage,Beverage_prep,Calories,Total Fat (g),Trans Fat (g),Saturated Fat (g),Sodium (mg),Total Carbohydrates (g),Cholesterol (mg),Dietary Fibre (g),Sugars (g),Protein (g),Vitamin A (% DV),Vitamin C (% DV),Calcium (% DV),Iron (% DV),Caffeine (mg)
0,Coffee,Brewed Coffee,Short,3,0.1,0.0,0.0,0,5,0,0,0,0.3,0%,0%,0%,0%,175
1,Coffee,Brewed Coffee,Tall,4,0.1,0.0,0.0,0,10,0,0,0,0.5,0%,0%,0%,0%,260
2,Coffee,Brewed Coffee,Grande,5,0.1,0.0,0.0,0,10,0,0,0,1.0,0%,0%,0%,0%,330
3,Coffee,Brewed Coffee,Venti,5,0.1,0.0,0.0,0,10,0,0,0,1.0,0%,0%,2%,0%,410
4,Classic Espresso Drinks,Caffè Latte,Short Nonfat Milk,70,0.1,0.1,0.0,5,75,10,0,9,6.0,10%,0%,20%,0%,75


In [4]:
stb.isna().sum()

Beverage_category            0
Beverage                     0
Beverage_prep                0
Calories                     0
 Total Fat (g)               0
Trans Fat (g)                0
Saturated Fat (g)            0
 Sodium (mg)                 0
 Total Carbohydrates (g)     0
Cholesterol (mg)             0
 Dietary Fibre (g)           0
 Sugars (g)                  0
 Protein (g)                 0
Vitamin A (% DV)             0
Vitamin C (% DV)             0
 Calcium (% DV)              0
Iron (% DV)                  0
Caffeine (mg)                1
dtype: int64

In [5]:
stb.dtypes

Beverage_category             object
Beverage                      object
Beverage_prep                 object
Calories                       int64
 Total Fat (g)                object
Trans Fat (g)                float64
Saturated Fat (g)            float64
 Sodium (mg)                   int64
 Total Carbohydrates (g)       int64
Cholesterol (mg)               int64
 Dietary Fibre (g)             int64
 Sugars (g)                    int64
 Protein (g)                 float64
Vitamin A (% DV)              object
Vitamin C (% DV)              object
 Calcium (% DV)               object
Iron (% DV)                   object
Caffeine (mg)                 object
dtype: object

1. **Examine and Rename Columns**
2. **Remove Percentage Signs**

In [6]:
stb.columns = (
    stb.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "")
    .str.replace(")", "")
    .str.replace("%", "percent")
)

In [7]:
stb = stb.rename(columns={
    'dietary_fibre_g': 'dietary_fiber_g'
})

3. **Convert to Number Types**

In [8]:
cols_to_convert = [
    'calories',
    'total_fat_g',
    'trans_fat_g',
    'saturated_fat_g',
    'sodium_mg',
    'total_carbohydrates_g',
    'cholesterol_mg',
    'dietary_fiber_g',
    'sugars_g',
    'protein_g',
    'vitamin_a_percent_dv',
    'vitamin_c_percent_dv',
    'calcium_percent_dv',
    'iron_percent_dv',
    'caffeine_mg'
]
stb[cols_to_convert] = stb[cols_to_convert].apply(
    pd.to_numeric, errors='coerce'
)
stb[cols_to_convert] = stb[cols_to_convert].round(2)

In [9]:
stb

,beverage_category,beverage,beverage_prep,calories,total_fat_g,trans_fat_g,saturated_fat_g,sodium_mg,total_carbohydrates_g,cholesterol_mg,dietary_fiber_g,sugars_g,protein_g,vitamin_a_percent_dv,vitamin_c_percent_dv,calcium_percent_dv,iron_percent_dv,caffeine_mg
0,Coffee,Brewed Coffee,Short,3,0.1,0.0,0.0,0,5,0,0,0,0.3,NaN,NaN,NaN,NaN,175.0
1,Coffee,Brewed Coffee,Tall,4,0.1,0.0,0.0,0,10,0,0,0,0.5,NaN,NaN,NaN,NaN,260.0
2,Coffee,Brewed Coffee,Grande,5,0.1,0.0,0.0,0,10,0,0,0,1.0,NaN,NaN,NaN,NaN,330.0
3,Coffee,Brewed Coffee,Venti,5,0.1,0.0,0.0,0,10,0,0,0,1.0,NaN,NaN,NaN,NaN,410.0
4,Classic Espresso Drinks,Caffè Latte,Short Nonfat Milk,70,0.1,0.1,0.0,5,75,10,0,9,6.0,NaN,NaN,NaN,NaN,75.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,Frappuccino® Blended Crème,Strawberries & Crème (Without Whipped Cream),Soymilk,320,NaN,0.4,0.0,0,250,67,1,64,5.0,NaN,NaN,NaN,NaN,0.0
238,Frappuccino® Blended Crème,Vanilla Bean (Without Whipped Cream),Tall Nonfat Milk,170,0.1,0.1,0.0,0,160,39,0,38,4.0,NaN,NaN,NaN,NaN,0.0
239,Frappuccino® Blended Crème,Vanilla Bean (Without Whipped Cream),Whole Milk,200,3.5,2.0,0.1,10,160,39,0,38,3.0,NaN,NaN,NaN,NaN,0.0
240,Frappuccino® Blended Crème,Vanilla Bean (Without Whipped Cream),Soymilk,180,1.5,0.2,0.0,0,160,37,1,35,3.0,NaN,NaN,NaN,NaN,0.0


In [10]:
stb.isna().sum()

beverage_category          0
beverage                   0
beverage_prep              0
calories                   0
total_fat_g                1
trans_fat_g                0
saturated_fat_g            0
sodium_mg                  0
total_carbohydrates_g      0
cholesterol_mg             0
dietary_fiber_g            0
sugars_g                   0
protein_g                  0
vitamin_a_percent_dv     242
vitamin_c_percent_dv     242
calcium_percent_dv       242
iron_percent_dv          242
caffeine_mg               23
dtype: int64

In [11]:
stb[stb['total_fat_g'].isna()]

,beverage_category,beverage,beverage_prep,calories,total_fat_g,trans_fat_g,saturated_fat_g,sodium_mg,total_carbohydrates_g,cholesterol_mg,dietary_fiber_g,sugars_g,protein_g,vitamin_a_percent_dv,vitamin_c_percent_dv,calcium_percent_dv,iron_percent_dv,caffeine_mg
237,Frappuccino® Blended Crème,Strawberries & Crème (Without Whipped Cream),Soymilk,320,NaN,0.4,0.0,0,250,67,1,64,5.0,NaN,NaN,NaN,NaN,0.0


In [12]:
stb['total_fat_g'] = stb['total_fat_g'].fillna(0)

In [13]:
vitamin_cols = [
    'vitamin_a_percent_dv',
    'vitamin_c_percent_dv',
    'calcium_percent_dv',
    'iron_percent_dv'
]

stb[vitamin_cols] = stb[vitamin_cols].fillna(0)

In [14]:
stb['caffeine_mg'] = stb['caffeine_mg'].fillna(0)

In [15]:
stb.describe()

,calories,total_fat_g,trans_fat_g,saturated_fat_g,sodium_mg,total_carbohydrates_g,cholesterol_mg,dietary_fiber_g,sugars_g,protein_g,vitamin_a_percent_dv,vitamin_c_percent_dv,calcium_percent_dv,iron_percent_dv,caffeine_mg
count,242.000000,242.000000,242.000000,242.000000,242.000000,242.000000,242.000000,242.000000,242.000000,242.000000,242.0,242.0,242.0,242.0,242.000000
mean,193.871901,2.891736,1.307025,0.037603,6.363636,128.884298,35.991736,0.805785,32.962810,6.978512,0.0,0.0,0.0,0.0,81.012397
std,102.863303,2.950226,1.640259,0.071377,8.630257,82.303223,20.795186,1.445944,19.730199,4.871659,0.0,0.0,0.0,0.0,66.946655
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000
25%,120.000000,0.200000,0.100000,0.000000,0.000000,70.000000,21.000000,0.000000,18.000000,3.000000,0.0,0.0,0.0,0.0,11.250000
50%,185.000000,2.250000,0.500000,0.000000,5.000000,125.000000,34.000000,0.000000,32.000000,6.000000,0.0,0.0,0.0,0.0,75.000000
75%,260.000000,4.500000,2.000000,0.100000,10.000000,170.000000,50.750000,1.000000,43.750000,10.000000,0.0,0.0,0.0,0.0,130.000000
max,510.000000,15.000000,9.000000,0.300000,40.000000,340.000000,90.000000,8.000000,84.000000,20.000000,0.0,0.0,0.0,0.0,410.000000


# Use the clean data for the Streamlit App

In [16]:
stb.to_csv("starbucks_clean.csv", index=False)

**## Streamlit App Development**